# Skin Health Monitor - Model Training
## For Diverse Indian Skin Tones

This notebook trains a MobileNetV3 model optimized for detecting common skin conditions on Indian skin tones.

**Steps:**
1. Setup environment
2. Download datasets
3. Preprocess images
4. Train model
5. Convert to TensorFlow Lite
6. Download for Android app

## Step 1: Setup Environment

In [ ]:
# Install dependencies
!pip install -q tensorflow tensorflow-datasets pillow numpy pandas scikit-learn matplotlib seaborn
!pip install -q kaggle

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import os

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## Step 2: Download Dataset

Upload your `kaggle.json` file when prompted.

In [ ]:
# Upload Kaggle API credentials
from google.colab import files
uploaded = files.upload()

# Setup Kaggle
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download Indian Skin Disease Dataset
!kaggle datasets download -d divypatel2000/indian-skin-disease-dataset
!unzip -q indian-skin-disease-dataset.zip -d indian_skin_data/

print("\nDataset downloaded successfully!")
print("\nDirectory structure:")
!ls -R indian_skin_data/ | head -20

## Step 3: Configure Classes and Data Preprocessing

In [ ]:
# Define your skin condition classes (adjust based on your dataset)
CLASSES = [
    'acne',
    'eczema',
    'tinea_fungal',
    'bacterial_infection',
    'vitiligo',
    'psoriasis',
    'contact_dermatitis',
    'viral_exanthem'
]

# Model configuration
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.001
FINE_TUNE_LR = 0.0001

print(f"Number of classes: {len(CLASSES)}")
print(f"Classes: {CLASSES}")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data augmentation for diverse lighting conditions
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.5, 1.5],  # Critical for varied lighting
    fill_mode='nearest',
    validation_split=0.2
)

# Test data (no augmentation)
test_datagen = ImageDataGenerator(
    rescale=1./255
)

# Load training data
train_data = train_datagen.flow_from_directory(
    'indian_skin_data/train',  # Adjust path based on your dataset structure
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

# Load validation data
val_data = train_datagen.flow_from_directory(
    'indian_skin_data/train',
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print(f"\nTraining samples: {train_data.samples}")
print(f"Validation samples: {val_data.samples}")
print(f"Class indices: {train_data.class_indices}")

## Step 4: Build MobileNetV3 Model

In [ ]:
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model

def build_model(num_classes):
    # Load pre-trained MobileNetV3-Large
    base_model = MobileNetV3Large(
        weights='imagenet',
        include_top=False,
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Build custom head
    inputs = Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.2)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model, base_model

# Build the model
model, base_model = build_model(len(CLASSES))

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')
    ]
)

model.summary()

## Step 5: Train Model (Transfer Learning)

In [ ]:
# Callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# Initial training with frozen base
print("Phase 1: Training with frozen base model...")
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
# Fine-tuning: Unfreeze last 20 layers
print("\nPhase 2: Fine-tuning last 20 layers...")
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')
    ]
)

# Continue training
history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
)

# Save final model
model.save('skin_health_model.h5')
print("\nModel saved as 'skin_health_model.h5'")

## Step 6: Visualize Training Results

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Print final metrics
print("\nFinal Training Metrics:")
print(f"Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"Training Loss: {history.history['loss'][-1]:.4f}")
print(f"Validation Loss: {history.history['val_loss'][-1]:.4f}")

## Step 7: Convert to TensorFlow Lite

In [ ]:
# Convert to TFLite with quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Post-training quantization for smaller size
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

# Convert
print("Converting to TFLite...")
tflite_model = converter.convert()

# Save TFLite model
with open('skin_health_model.tflite', 'wb') as f:
    f.write(tflite_model)

model_size_mb = len(tflite_model) / (1024 * 1024)
print(f"\nTFLite model created!")
print(f"Model size: {model_size_mb:.2f} MB")

# Save class labels
with open('labels.txt', 'w') as f:
    for label in CLASSES:
        f.write(f"{label}\n")

print("Labels saved to 'labels.txt'")

## Step 8: Test TFLite Model

In [ ]:
# Load TFLite model
interpreter = tf.lite.Interpreter(model_path='skin_health_model.tflite')
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite Model Details:")
print(f"Input shape: {input_details[0]['shape']}")
print(f"Input type: {input_details[0]['dtype']}")
print(f"Output shape: {output_details[0]['shape']}")
print(f"Output type: {output_details[0]['dtype']}")

# Test with validation data
import cv2
from PIL import Image

def test_tflite_model(image_path):
    # Load and preprocess image
    img = Image.open(image_path).convert('RGB')
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE))
    img_array = np.array(img).astype(np.float32) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Run inference
    interpreter.set_tensor(input_details[0]['index'], img_array)
    interpreter.invoke()
    predictions = interpreter.get_tensor(output_details[0]['index'])
    
    # Get prediction
    predicted_idx = np.argmax(predictions[0])
    confidence = predictions[0][predicted_idx]
    
    print(f"\nPrediction for {image_path}:")
    print(f"Class: {CLASSES[predicted_idx]}")
    print(f"Confidence: {confidence:.2%}")
    
    # Show top 3 predictions
    top_3_idx = np.argsort(predictions[0])[-3:][::-1]
    print("\nTop 3 predictions:")
    for idx in top_3_idx:
        print(f"  {CLASSES[idx]}: {predictions[0][idx]:.2%}")

# Test on a sample image (adjust path)
# test_tflite_model('path/to/test/image.jpg')

print("\nTFLite model is ready for Android integration!")

## Step 9: Download Files for Android App

In [ ]:
from google.colab import files

# Download TFLite model
print("Downloading skin_health_model.tflite...")
files.download('skin_health_model.tflite')

# Download labels
print("Downloading labels.txt...")
files.download('labels.txt')

# Optionally download full model
print("Downloading skin_health_model.h5 (optional)...")
files.download('skin_health_model.h5')

print("\n✓ All files downloaded!")
print("\nNext steps:")
print("1. Add skin_health_model.tflite to your Android app's assets folder")
print("2. Add labels.txt to your Android app's assets folder")
print("3. Follow the implementation guide for Android app development")

## Bonus: Performance Metrics and Bias Analysis

In [ ]:
# Evaluate on validation set
print("Evaluating on validation set...")
val_loss, val_accuracy, val_top3 = model.evaluate(val_data)

print(f"\nValidation Results:")
print(f"Loss: {val_loss:.4f}")
print(f"Accuracy: {val_accuracy:.4f}")
print(f"Top-3 Accuracy: {val_top3:.4f}")

# Get predictions for confusion matrix
val_data.reset()
predictions = model.predict(val_data, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = val_data.classes

# Confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(true_classes, predicted_classes,
                          target_names=CLASSES))